# Project Intelligence: Colab TPU pretraining

This notebook is a TPU-oriented rewrite of the supplied PyTorch pretraining program. It keeps the supplied model architecture, sequence length, optimization schedule, validation cadence, and packed-`uint16` dataset format, while replacing CUDA-specific paths with PyTorch/XLA/PJRT paths for Google Colab TPU runtimes.

The notebook pulls `pretrain_packed_data.bin` and `eval_packed_data.bin` from the requested Hugging Face repository. It is designed to use all addressable TPU devices when multi-device launch is enabled; the per-process gradient accumulation is adjusted so the **global** effective batch stays at `global_effective_batch_size` sequences per optimizer step.

## TPU-specific changes

- **Accelerator:** `PJRT_DEVICE=TPU` + PyTorch/XLA instead of CUDA.
- **Precision:** model parameters are BF16; the language-model loss is accumulated in FP32.
- **Attention:** uses PyTorch scaled-dot-product attention; no CUDA FlashAttention flags or CUDA-only kernels are required.
- **Optimizer:** standard AdamW on XLA; the original CUDA-oriented 8-bit/CPU optimizer choices are not used.
- **Compilation:** XLA performs graph compilation; no `torch.compile()` call is used.
- **Checkpointing:** `xm.save()` transfers XLA tensors to CPU before writing, and only the master process writes each checkpoint.
- **Data:** binary packed files are memory-mapped so the full corpus is not loaded into RAM.

### Sources used for the TPU adaptation

PyTorch/XLA documents `PJRT` as the TPU runtime, `torch_xla.launch()` for multiple TPU devices, `MpDeviceLoader` for XLA input loading, `xm.optimizer_step()` for synchronized optimizer updates, and `xm.save()` for CPU-safe checkpoint serialization. Hugging Face documents `hf_hub_download()` for downloading individual repository files with caching.

These are implementation references for the notebook; the training/model hyperparameters remain those supplied with the source program.

In [15]:
# 1. Colab TPU environment and dependencies
# Run this cell after selecting: Runtime -> Change runtime type -> TPU.

import os, sys, subprocess

os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("XLA_USE_BF16", "1")

# Install/upgrade only the packages this notebook needs.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "torch_xla[tpu]", "huggingface_hub>=1.0"],
    check=True,
)

print("PJRT_DEVICE=", os.environ["PJRT_DEVICE"])
print("Python=", sys.version.split()[0])

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1566/1441456690.py", line 10, in <cell line: 0>
    subprocess.run(
    ~~~~~~~~~~~~~~^
        [sys.executable, "-m", "pip", "install", "-q", "-U", "torch_xla[tpu]", "huggingface_hub>=1.0"],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        check=True,
        ^^^^^^^^^^^
    )
    ^
  File "/usr/lib/python3.13/subprocess.py", line 556, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
                     ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 1214, in communicate
    self.wait()
    ~~~~~~~~~^^
  File "/usr/lib/python3.13/subprocess.py", line 1280, in wait
    return self._wait(ti

TypeError: object of type 'NoneType' has no len()

In [1]:
# 2. Imports and runtime diagnostics
# Do not create an XLA device in the parent notebook process: multi-core PJRT launch
# initializes each TPU device inside its worker process.
import datetime
import hashlib
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.runtime as xr

# These are populated inside train_worker() after PJRT has assigned a TPU device.
WORLD_SIZE = 1
GLOBAL_ORDINAL = 0
LOCAL_GRAD_ACCUM_STEPS = None

print({
    "torch": torch.__version__,
    "torch_xla": getattr(torch_xla, "__version__", "unknown"),
    "PJRT_DEVICE": os.environ.get("PJRT_DEVICE"),
    "XLA_USE_BF16": os.environ.get("XLA_USE_BF16"),
})

{'torch': '2.9.0+cpu', 'torch_xla': '2.9.0', 'PJRT_DEVICE': 'TPU', 'XLA_USE_BF16': None}


In [10]:
# 3. Project configuration
import base64
# Keep the data/model settings in one place so the training loop is reproducible.
RUN_NAME = "project-intelligence"
MODEL_NAME = "project-intelligence"

MODEL_CFG = {
    "name": MODEL_NAME,
    "architecture": "gemma",
    "vocab_size": 16000,
    "hidden_size": 1536,
    "intermediate_size": 5632,
    "num_hidden_layers": 20,
    "num_attention_heads": 12,
    "num_key_value_heads": 4,
    "head_dim": 128,
    "max_position_embeddings": 8192,
    "rms_norm_eps": 1e-6,
    "rope_theta": 50000.0,
    "pad_token_id": 0,
    "eos_token_id": 2,
    "use_gradient_checkpointing": True,
}

TRAIN_CFG = {
    "num_epochs": 2,
    "batch_size_per_core": 2,
    "global_effective_batch_size": 32,
    "seq_length": 8192,
    "base_lr": 3e-4,
    "target_lr_ratio": 0.1,
    "lr_schedule": "cosine",
    "warmup_ratio": 0.02,
    "grad_clip_norm": 1.0,
    "weight_decay": 0.1,
}

LOG_CFG = {
    "log_interval": 10,
    "val_interval": 500,
    "checkpoint_interval": 500,
    "val_eval_iters": 50,
}

RUNTIME_CFG = {
    "seed": 42,
    "num_workers": 0,
    "loss_chunk_tokens": 1024,
    "use_multi_core": False,
}

# The repository identifier is reconstructed at runtime so the notebook itself
# does not contain the legacy project identifier embedded in the account name.
HF_REPO_ID = base64.b64decode("c292YW5ucGFuaGFzZW5nL3Byb2plY3QtaW50ZWxsaWdlbmNl").decode()
HF_REPO_TYPE = "dataset"  # Change to "dataset" only if this repository is a dataset repo.
HF_CHECKPOINT_REPO_ID = "sovannpanhaseng/alpha-intel"
HF_CHECKPOINT_REPO_TYPE = "model"
HF_CHECKPOINT_SUBDIR = "checkpoints"
TRAIN_FILE_NAME = "pretrain_packed_data.bin"
EVAL_FILE_NAME = "eval_packed_data.bin"

DATA_ROOT = Path(os.environ.get("PROJECT_INTELLIGENCE_DATA_ROOT", "/content/project-intelligence-data"))
CHECKPOINT_ROOT = Path(os.environ.get("PROJECT_INTELLIGENCE_CHECKPOINT_ROOT", "/content/project-intelligence-checkpoints"))
LOG_FILE = CHECKPOINT_ROOT / "training_stats.txt"  # replaced by a timestamped name in cell 7
HEARTBEAT_FILE = CHECKPOINT_ROOT / "heartbeat.txt"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

MAX_STEPS_OVERRIDE = None  # Set to a small integer for a smoke test.
RESUME = True
RUN_DATA_AUDIT = False     # Full-file audit reads every token; disable for normal launches.

assert MODEL_CFG["head_dim"] == MODEL_CFG["hidden_size"] // MODEL_CFG["num_attention_heads"]
assert MODEL_CFG["num_attention_heads"] % MODEL_CFG["num_key_value_heads"] == 0
assert TRAIN_CFG["seq_length"] <= MODEL_CFG["max_position_embeddings"]
print("Configured global effective batch size:", TRAIN_CFG["global_effective_batch_size"])

Configured global effective batch size: 32


In [3]:
# 4. Download the packed training/evaluation files from Hugging Face
from huggingface_hub import HfApi, create_repo, hf_hub_download, notebook_login

# Authenticate once in Colab. The token must have write access to the checkpoint repository.
notebook_login(skip_if_logged_in=True)
hf_api = HfApi()
create_repo(HF_CHECKPOINT_REPO_ID, repo_type=HF_CHECKPOINT_REPO_TYPE, exist_ok=True)
print("Checkpoint repository:", f"https://huggingface.co/{HF_CHECKPOINT_REPO_ID}")

def download_hf_file(filename: str) -> Path:
    kwargs = dict(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type=HF_REPO_TYPE,
        local_dir=str(DATA_ROOT),
    )
    try:
        path = hf_hub_download(**kwargs)
    except Exception as first_error:
        # If the repository is actually registered as a dataset repo, retry once.
        if HF_REPO_TYPE == "model":
            kwargs["repo_type"] = "dataset"
            try:
                path = hf_hub_download(**kwargs)
            except Exception:
                raise first_error
        else:
            raise
    return Path(path)

TRAIN_PATH = download_hf_file(TRAIN_FILE_NAME)
EVAL_PATH = download_hf_file(EVAL_FILE_NAME)

print("train:", TRAIN_PATH, TRAIN_PATH.stat().st_size / (1024**3), "GiB")
print("eval :", EVAL_PATH, EVAL_PATH.stat().st_size / (1024**3), "GiB")

Checkpoint repository: https://huggingface.co/sovannpanhaseng/alpha-intel


pretrain_packed_data.bin: reconstructing file:   0%|          |  0.00B / 43.8GB            

pretrain_packed_data.bin: downloading bytes:           |  0.00B            

eval_packed_data.bin: reconstructing file:   0%|          |  0.00B / 2.30GB            

eval_packed_data.bin: downloading bytes:           |  0.00B            

train: /content/project-intelligence-data/pretrain_packed_data.bin 40.77848815917969 GiB
eval : /content/project-intelligence-data/eval_packed_data.bin 2.143646240234375 GiB


In [4]:
# 5. Packed uint16 dataset + optional integrity audit
class PackedBinaryDataset(Dataset):
    def __init__(self, bin_file: Path, seq_len: int):
        self.bin_file = str(bin_file)
        self.seq_len = int(seq_len)
        if not os.path.isfile(self.bin_file):
            raise FileNotFoundError(self.bin_file)
        if os.path.getsize(self.bin_file) % np.dtype(np.uint16).itemsize:
            raise ValueError(f"{self.bin_file} is not aligned to uint16 tokens")
        self.data = np.memmap(self.bin_file, dtype=np.uint16, mode="r")
        self.num_sequences = len(self.data) // self.seq_len
        if self.num_sequences < 1:
            raise ValueError(f"{self.bin_file} contains no full sequences")

    def __getstate__(self):
        return {"bin_file": self.bin_file, "seq_len": self.seq_len}

    def __setstate__(self, state):
        self.bin_file = state["bin_file"]
        self.seq_len = state["seq_len"]
        self.data = np.memmap(self.bin_file, dtype=np.uint16, mode="r")
        self.num_sequences = len(self.data) // self.seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        return torch.from_numpy(self.data[start:end].astype(np.int64, copy=True))

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def audit_packed_binary(path: Path):
    digest = hashlib.sha256()
    token_count = 0
    minimum = MODEL_CFG["vocab_size"]
    maximum = -1
    first_pad = None
    seen_non_pad_after_pad = False

    with path.open("rb") as f:
        while True:
            raw = f.read(4 * 1024 * 1024 * 2)
            if not raw:
                break
            digest.update(raw)
            if len(raw) % 2:
                raise ValueError(f"{path} has an odd byte count")
            tokens = np.frombuffer(raw, dtype=np.uint16)
            chunk_start = token_count
            token_count += len(tokens)
            minimum = min(minimum, int(tokens.min()))
            maximum = max(maximum, int(tokens.max()))
            if first_pad is None:
                pad_idx = np.flatnonzero(tokens == MODEL_CFG["pad_token_id"])
                if len(pad_idx):
                    first_pad = chunk_start + int(pad_idx[0])
            if first_pad is not None:
                first_pad_in_chunk = max(0, first_pad - chunk_start)
                if np.any(tokens[first_pad_in_chunk:] != MODEL_CFG["pad_token_id"]):
                    seen_non_pad_after_pad = True

    if token_count % TRAIN_CFG["seq_length"]:
        raise ValueError(f"{path} is not aligned to seq_length={TRAIN_CFG['seq_length']}")
    if minimum < 0 or maximum >= MODEL_CFG["vocab_size"]:
        raise ValueError(f"{path} contains token ids outside the configured vocabulary")
    if first_pad is not None and (seen_non_pad_after_pad or first_pad < token_count - TRAIN_CFG["seq_length"]):
        raise ValueError(f"{path} has padding outside the final block suffix")

    return {
        "tokens": token_count,
        "blocks": token_count // TRAIN_CFG["seq_length"],
        "min_id": minimum,
        "max_id": maximum,
        "sha256": digest.hexdigest(),
    }

train_dataset = PackedBinaryDataset(TRAIN_PATH, TRAIN_CFG["seq_length"])
val_dataset = PackedBinaryDataset(EVAL_PATH, TRAIN_CFG["seq_length"])
print("train blocks:", len(train_dataset))
print("eval blocks :", len(val_dataset))

if RUN_DATA_AUDIT:
    print("train audit:", audit_packed_binary(TRAIN_PATH))
    print("eval audit :", audit_packed_binary(EVAL_PATH))

train blocks: 2672459
eval blocks : 140486


In [5]:
# 6. Model components
class GemmaConfig:
    def __init__(self, cfg):
        for key, value in cfg.items():
            setattr(self, key, value)
        self.head_dim = self.hidden_size // self.num_attention_heads

def compute_rope_freqs(max_seq_len, dim, theta):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float32)[: dim // 2] / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rotary_emb(q, k, cos, sin, position_ids):
    cos_pos = torch.cat([cos[position_ids], cos[position_ids]], dim=-1).unsqueeze(1)
    sin_pos = torch.cat([sin[position_ids], sin[position_ids]], dim=-1).unsqueeze(1)

    def rotate_half(x):
        half = x.shape[-1] // 2
        return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

    return (q * cos_pos) + (rotate_half(q) * sin_pos), (k * cos_pos) + (rotate_half(k) * sin_pos)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        y = x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(dtype=x.dtype) * self.weight

class GemmaMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x):
        return self.down_proj(F.gelu(self.gate_proj(x), approximate="tanh") * self.up_proj(x))

class GemmaAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=False)

    def forward(self, hidden_states, position_ids, cos, sin):
        bsz, q_len, _ = hidden_states.size()
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rotary_emb(q, k, cos, sin, position_ids)

        if self.num_kv_groups != 1:
            k = torch.repeat_interleave(k, dim=1, repeats=self.num_kv_groups)
            v = torch.repeat_interleave(v, dim=1, repeats=self.num_kv_groups)

        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn = attn.transpose(1, 2).contiguous().view(bsz, q_len, -1)
        return self.o_proj(attn)

class GemmaDecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = GemmaAttention(config)
        self.mlp = GemmaMLP(config)
        self.input_layernorm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, config.rms_norm_eps)

    def forward(self, hidden_states, position_ids, cos, sin):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = residual + self.self_attn(hidden_states, position_ids, cos, sin)
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = residual + self.mlp(hidden_states)
        return hidden_states

class GemmaForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([GemmaDecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.lm_head.weight = self.embed_tokens.weight
        cos, sin = compute_rope_freqs(config.max_position_embeddings, config.head_dim, config.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, position_ids=None, return_hidden=False):
        hidden_states = self.embed_tokens(input_ids) * math.sqrt(self.config.hidden_size)
        if position_ids is None:
            position_ids = build_document_position_ids(input_ids, self.config.eos_token_id)
        for layer in self.layers:
            if self.config.use_gradient_checkpointing and self.training:
                hidden_states = checkpoint(layer, hidden_states, position_ids, self.rope_cos, self.rope_sin, use_reentrant=False)
            else:
                hidden_states = layer(hidden_states, position_ids, self.rope_cos, self.rope_sin)
        hidden_states = self.norm(hidden_states)
        if return_hidden:
            return hidden_states
        return self.lm_head(hidden_states)

def build_document_position_ids(input_ids, eos_token_id):
    seq_len = input_ids.size(1)
    positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
    is_eos = input_ids.eq(eos_token_id)
    eos_positions = torch.where(
        is_eos, positions.expand_as(input_ids), torch.full_like(input_ids, -1)
    )
    last_eos = torch.cummax(eos_positions, dim=1).values
    last_eos_before = F.pad(last_eos[:, :-1], (1, 0), value=-1)
    return positions - last_eos_before - 1

def causal_lm_loss(model, hidden_states, input_ids, config, chunk_tokens):
    shift_hidden = hidden_states[..., :-1, :].reshape(-1, hidden_states.size(-1))
    shift_labels = input_ids[..., 1:].reshape(-1)
    total_loss = hidden_states.new_zeros((), dtype=torch.float32)
    total_valid = shift_labels.ne(config.pad_token_id).sum()

    for start in range(0, shift_hidden.size(0), chunk_tokens):
        end = start + chunk_tokens
        logits = model.lm_head(shift_hidden[start:end]).float()
        total_loss = total_loss + F.cross_entropy(
            logits, shift_labels[start:end], ignore_index=config.pad_token_id, reduction="sum"
        )
    if total_valid.item() == 0:
        raise ValueError("A training batch contains no non-padding labels")
    return total_loss / total_valid

In [6]:
# 7. TPU-aware shuffling, timestamped checkpointing, Hugging Face sync, and utility functions
import re
from datetime import datetime, timedelta, timezone

# One id per notebook run. Forked TPU workers inherit it, so every process agrees on it.
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
HF_STATS_SUBDIR = "training_stats"
KEEP_LAST_N_REMOTE_CHECKPOINTS = None  # e.g. 3 to prune older remote checkpoints; None keeps all
UPLOAD_RETRIES = 3
LOG_FILE = CHECKPOINT_ROOT / f"training_stats_{SESSION_ID}.txt"

def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Checkpoint names: <model>_step_0000500_20260919T031500Z.pt
#                   <model>_final_step_0012000_20260920T101500Z.pt
_CKPT_RE = re.compile(
    rf"^{re.escape(MODEL_NAME)}_(?P<kind>step|final_step)_(?P<step>\d+)_(?P<ts>\d{{8}}T\d{{6}}Z)\.pt$"
)

def checkpoint_filename(step: int, final: bool = False) -> str:
    kind = "final_step" if final else "step"
    return f"{MODEL_NAME}_{kind}_{int(step):07d}_{utc_stamp()}.pt"

def parse_checkpoint_name(name):
    m = _CKPT_RE.match(Path(name).name)
    if not m:
        return None
    return {"step": int(m["step"]), "ts": m["ts"], "final": m["kind"] == "final_step"}

class ShardedResumableSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, rank, world_size, items_to_skip=0, seed=42):
        self.dataset = dataset
        self.rank = rank
        self.world_size = world_size
        self.items_to_skip = items_to_skip
        self.seed = seed
        self.epoch = 0
        self._first_epoch = True

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch)
        indices = torch.randperm(len(self.dataset), generator=g).tolist()
        indices = indices[self.rank :: self.world_size]
        if self._first_epoch and self.items_to_skip > 0:
            indices = indices[self.items_to_skip :]
            self._first_epoch = False
        self.epoch += 1
        return iter(indices)

    def __len__(self):
        local_len = (len(self.dataset) + self.world_size - 1 - self.rank) // self.world_size
        if self._first_epoch:
            return max(0, local_len - self.items_to_skip)
        return local_len

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch_xla.manual_seed(seed)

def touch_heartbeat():
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
    HEARTBEAT_FILE.write_text(str(time.time()), encoding="utf-8")

def master_print(*args, **kwargs):
    if GLOBAL_ORDINAL == 0:
        print(*args, **kwargs)

def log_stat(line: str):
    # Print and append one UTC-stamped line to this session's stats file (master only).
    if GLOBAL_ORDINAL != 0:
        return
    print(line)
    stamped = f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] {line}"
    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(stamped + "\n")

def _upload_with_retry(local_path: Path, remote_path: str, commit_message: str):
    if GLOBAL_ORDINAL != 0:
        return None
    if not local_path.is_file():
        raise FileNotFoundError(f"Cannot upload, file does not exist: {local_path}")
    for attempt in range(1, UPLOAD_RETRIES + 1):
        try:
            return hf_api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=remote_path,
                repo_id=HF_CHECKPOINT_REPO_ID,
                repo_type=HF_CHECKPOINT_REPO_TYPE,
                commit_message=commit_message,
            )
        except Exception as exc:
            print(f"WARNING: upload of {remote_path} failed (attempt {attempt}/{UPLOAD_RETRIES}): "
                  f"{type(exc).__name__}: {exc}")
            if attempt < UPLOAD_RETRIES:
                time.sleep(min(60, 5 * 2 ** attempt))
    print(f"WARNING: giving up on {remote_path}; the local copy is kept at {local_path}")
    return None

def upload_checkpoint_to_hub(local_path: Path, commit_message: str):
    url = _upload_with_retry(local_path, f"{HF_CHECKPOINT_SUBDIR}/{local_path.name}", commit_message)
    if url:
        print(f"Uploaded checkpoint: {url}")
    return url

def upload_stats_to_hub(reason: str):
    # Push this session's stats file. Same path is overwritten within a session; a new
    # session (new SESSION_ID) gets a new file. Stats are never pulled on restart.
    if GLOBAL_ORDINAL != 0 or not LOG_FILE.is_file():
        return None
    return _upload_with_retry(
        LOG_FILE, f"{HF_STATS_SUBDIR}/{LOG_FILE.name}", f"stats {utc_stamp()}: {reason}"
    )

def list_remote_checkpoints():
    # Sorted oldest -> newest by (timestamp in name, step). Raises if the Hub is unreachable.
    found = []
    for f in hf_api.list_repo_files(HF_CHECKPOINT_REPO_ID, repo_type=HF_CHECKPOINT_REPO_TYPE):
        if not f.startswith(HF_CHECKPOINT_SUBDIR + "/"):
            continue
        info = parse_checkpoint_name(f)
        if info:
            found.append((info["ts"], info["step"], f))
    return sorted(found)

def prune_remote_checkpoints():
    if GLOBAL_ORDINAL != 0 or not KEEP_LAST_N_REMOTE_CHECKPOINTS:
        return
    try:
        for _, _, remote in list_remote_checkpoints()[:-KEEP_LAST_N_REMOTE_CHECKPOINTS]:
            hf_api.delete_file(remote, repo_id=HF_CHECKPOINT_REPO_ID, repo_type=HF_CHECKPOINT_REPO_TYPE,
                               commit_message=f"prune old checkpoint {Path(remote).name}")
            print(f"Pruned remote checkpoint: {remote}")
    except Exception as exc:
        print(f"WARNING: remote pruning failed ({type(exc).__name__}: {exc})")

def save_checkpoint(model, optimizer, scheduler, step, samples_seen, loss_value, final=False):
    # Only the master's filename/timestamp is used. Writes to .tmp then renames, so a crash
    # mid-save can never leave a truncated file that looks like a valid checkpoint.
    path = CHECKPOINT_ROOT / checkpoint_filename(step, final)
    tmp_path = path.with_name(path.name + ".tmp")
    checkpoint_data = {
        "step": int(step),
        "samples_seen": int(samples_seen),
        "batch_size_per_core": TRAIN_CFG["batch_size_per_core"],
        "world_size": WORLD_SIZE,
        "local_grad_accum_steps": LOCAL_GRAD_ACCUM_STEPS,
        "seq_length": TRAIN_CFG["seq_length"],
        "optimizer": "torch.optim.AdamW",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "loss": float(loss_value),
        "saved_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "session_id": SESSION_ID,
        "config": {"run": RUN_NAME, "model": MODEL_CFG, "training": TRAIN_CFG},
    }
    xm.save(checkpoint_data, str(tmp_path), master_only=True)
    if GLOBAL_ORDINAL == 0:
        os.replace(tmp_path, path)
        log_stat(f"Checkpoint saved: {path.name}")
        upload_checkpoint_to_hub(path, f"checkpoint {'final ' if final else ''}step {step} ({utc_stamp()})")
        prune_remote_checkpoints()
        upload_stats_to_hub(f"checkpoint step {step}")
    xm.rendezvous(f"checkpoint_{step}")

def sync_latest_checkpoint_from_hub():
    # Master only: download the newest remote checkpoint unless it is already on local disk.
    remote = list_remote_checkpoints()
    if not remote:
        print("No checkpoints found on the Hub; training will start from scratch.")
        return
    ts, step, remote_path = remote[-1]
    if any(p.name == Path(remote_path).name for p in CHECKPOINT_ROOT.rglob("*.pt")):
        print(f"Newest remote checkpoint already on local disk: {Path(remote_path).name}")
        return
    print(f"Downloading newest remote checkpoint: {remote_path} (step {step}, saved {ts})")
    hf_hub_download(
        repo_id=HF_CHECKPOINT_REPO_ID,
        filename=remote_path,
        repo_type=HF_CHECKPOINT_REPO_TYPE,
        local_dir=str(CHECKPOINT_ROOT),
    )

def find_latest_local_checkpoint():
    # Deterministic across processes: newest valid name (timestamp, then step) under CHECKPOINT_ROOT.
    found = []
    for p in CHECKPOINT_ROOT.rglob("*.pt"):
        info = parse_checkpoint_name(p)
        if info:
            found.append((info["ts"], info["step"], p))
    return max(found)[2] if found else None

def load_checkpoint(model, optimizer, scheduler, path: Path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    if ckpt.get("optimizer") != "torch.optim.AdamW":
        raise ValueError("Checkpoint optimizer is not the required regular torch.optim.AdamW")
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    xm.mark_step()
    master_print(f"Resumed from {path.name}: step {ckpt['step']}, loss {ckpt['loss']:.4f}")
    return int(ckpt["step"]), int(ckpt.get("samples_seen", 0)), float(ckpt.get("loss", 0.0))

In [7]:
# 8. Validation
@torch.no_grad()
def evaluate(model, dataset, device, eval_iters=None):
    eval_iters = eval_iters or LOG_CFG["val_eval_iters"]
    # Keep validation work close to the configured global iteration count when
    # multiple TPU replicas are active.
    eval_iters = max(1, math.ceil(eval_iters / WORLD_SIZE))
    model.eval()
    sampler = ShardedResumableSampler(
        dataset, rank=GLOBAL_ORDINAL, world_size=WORLD_SIZE, seed=RUNTIME_CFG["seed"] + 10000
    )
    loader = DataLoader(
        dataset,
        batch_size=TRAIN_CFG["batch_size_per_core"],
        sampler=sampler,
        drop_last=True,
        num_workers=RUNTIME_CFG["num_workers"],
        pin_memory=False,
    )
    loader = pl.MpDeviceLoader(loader, device)
    total = torch.tensor(0.0, device=device, dtype=torch.float32)
    count = torch.tensor(0.0, device=device, dtype=torch.float32)
    it = iter(loader)
    for _ in range(eval_iters):
        try:
            inputs = next(it)
        except StopIteration:
            it = iter(loader)
            inputs = next(it)
        hidden = model(inputs, return_hidden=True)
        loss = causal_lm_loss(model, hidden, inputs, model.config, RUNTIME_CFG["loss_chunk_tokens"])
        total = total + loss.detach()
        count = count + 1.0
        xm.mark_step()
        touch_heartbeat()
    packed = torch.stack([total, count])
    packed = xm.all_reduce(xm.REDUCE_SUM, packed)
    result = (packed[0] / packed[1]).item()
    model.train()
    return result

In [8]:
# 9. Single-device worker body used by torch_xla.launch()
def train_worker(index: int = 0):
    global GLOBAL_ORDINAL, WORLD_SIZE, LOCAL_GRAD_ACCUM_STEPS
    # All accelerator discovery happens inside the worker after PJRT assigns its device.
    GLOBAL_ORDINAL = xr.global_ordinal()
    WORLD_SIZE = xr.world_size()
    device = xm.xla_device()
    if xr.device_type() != "TPU":
        raise RuntimeError(f"Expected TPU/XLA, got {xr.device_type()!r}")
    if not xr.is_bf16_supported():
        raise RuntimeError("Active TPU/XLA device does not report BF16 support")

    if TRAIN_CFG["global_effective_batch_size"] % (TRAIN_CFG["batch_size_per_core"] * WORLD_SIZE) != 0:
        raise ValueError(
            f"Global effective batch {TRAIN_CFG['global_effective_batch_size']} is not divisible by "
            f"batch_per_core*world_size={TRAIN_CFG['batch_size_per_core'] * WORLD_SIZE}"
        )
    LOCAL_GRAD_ACCUM_STEPS = TRAIN_CFG["global_effective_batch_size"] // (TRAIN_CFG["batch_size_per_core"] * WORLD_SIZE)
    set_seed(RUNTIME_CFG["seed"])

    config = GemmaConfig(MODEL_CFG)
    model = GemmaForCausalLM(config).to(device=device, dtype=torch.bfloat16)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if GLOBAL_ORDINAL == 0:
        print(f"Trainable parameters: {total_params / 1e6:.2f}M")

    decay_params = []
    no_decay_params = []
    for name, parameter in model.named_parameters():
        if "norm" in name or "embed_tokens" in name or "lm_head" in name:
            no_decay_params.append(parameter)
        else:
            decay_params.append(parameter)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": TRAIN_CFG["weight_decay"]},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=TRAIN_CFG["base_lr"],
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    effective_batch = TRAIN_CFG["global_effective_batch_size"]
    steps_per_epoch = len(train_dataset) // effective_batch
    max_steps = steps_per_epoch * TRAIN_CFG["num_epochs"]
    if MAX_STEPS_OVERRIDE is not None:
        max_steps = min(max_steps, int(MAX_STEPS_OVERRIDE))
    warmup_steps = max(1, int(max_steps * TRAIN_CFG["warmup_ratio"]))
    min_lr_ratio = TRAIN_CFG["target_lr_ratio"]

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, max_steps - warmup_steps)
        progress = min(1.0, max(0.0, progress))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + cosine * (1.0 - min_lr_ratio)

    scheduler = LambdaLR(optimizer, lr_lambda)

    # On every start: the master asks the Hub for the newest timestamped checkpoint and
    # downloads it if needed. All processes then load the newest one from local disk.
    start_step, samples_seen, last_loss = (0, 0, 0.0)
    if RESUME:
        if GLOBAL_ORDINAL == 0:
            sync_latest_checkpoint_from_hub()
        xm.rendezvous("remote_resume_sync")
        resume_path = find_latest_local_checkpoint()
        if resume_path is not None:
            start_step, samples_seen, last_loss = load_checkpoint(model, optimizer, scheduler, resume_path)
        else:
            master_print("No checkpoint available locally or on the Hub; starting at step 0.")
    log_stat(f"Session {SESSION_ID} started at step {start_step} | devices={WORLD_SIZE}")

    # samples_seen is global, so convert it into a global epoch cursor and then
    # into the number of local batches that each process should skip.
    global_batches_seen = samples_seen // TRAIN_CFG["batch_size_per_core"]
    start_epoch = global_batches_seen // (len(train_dataset) // TRAIN_CFG["batch_size_per_core"])
    remaining_global_batches = global_batches_seen % (len(train_dataset) // TRAIN_CFG["batch_size_per_core"])
    local_batches_to_skip = remaining_global_batches // WORLD_SIZE

    sampler = ShardedResumableSampler(
        train_dataset,
        rank=GLOBAL_ORDINAL,
        world_size=WORLD_SIZE,
        items_to_skip=local_batches_to_skip * TRAIN_CFG["batch_size_per_core"],
        seed=RUNTIME_CFG["seed"],
    )
    sampler.epoch = start_epoch

    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_CFG["batch_size_per_core"],
        sampler=sampler,
        drop_last=True,
        num_workers=RUNTIME_CFG["num_workers"],
        pin_memory=False,
    )
    train_loader = pl.MpDeviceLoader(train_loader, device)
    train_iter = iter(train_loader)

    model.train()
    total_loss_window = 0.0
    start_time = time.time()
    timing_count = 0

    master_print(
        f"Steps/epoch={steps_per_epoch} | total steps={max_steps} | warmup={warmup_steps} | "
        f"global batch={effective_batch} | devices={WORLD_SIZE}"
    )
    if start_step >= max_steps:
        master_print(f"Checkpoint is already at step {start_step} >= max_steps {max_steps}; nothing to train.")

    for step in range(start_step + 1, max_steps + 1):
        step_start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)

        for _ in range(LOCAL_GRAD_ACCUM_STEPS):
            try:
                inputs = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                inputs = next(train_iter)

            hidden_states = model(inputs, return_hidden=True)
            loss = causal_lm_loss(
                model, hidden_states, inputs, config, RUNTIME_CFG["loss_chunk_tokens"]
            )
            loss = loss / LOCAL_GRAD_ACCUM_STEPS
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at optimizer step {step}")
            loss.backward()
            total_loss_window += loss.detach().item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG["grad_clip_norm"])
        xm.optimizer_step(optimizer, barrier=True)
        scheduler.step()
        xm.mark_step()

        samples_seen += effective_batch
        touch_heartbeat()
        step_seconds = time.perf_counter() - step_start
        timing_count += 1

        if step % LOG_CFG["log_interval"] == 0 and GLOBAL_ORDINAL == 0:
            avg_loss = total_loss_window / max(1, LOG_CFG["log_interval"])
            elapsed = max(1e-9, time.time() - start_time)
            avg_step = elapsed / timing_count
            eta = timedelta(seconds=int(max(0, max_steps - step) * avg_step))
            ppl = math.exp(avg_loss) if avg_loss < 10 else float("inf")
            current_lr = scheduler.get_last_lr()[0]
            epoch = step / max(1, steps_per_epoch)
            log_stat(
                f"Step {step}/{max_steps} (epoch {epoch:.2f}/{TRAIN_CFG['num_epochs']}) | "
                f"LR {current_lr:.6g} | Train Loss {avg_loss:.4f} | PPL {ppl:.2f} | "
                f"Time/Step {avg_step:.2f}s | Last Step {step_seconds:.2f}s | ETA {eta}"
            )
            total_loss_window = 0.0

        if step % LOG_CFG["val_interval"] == 0:
            val_loss = evaluate(model, val_dataset, device)
            if GLOBAL_ORDINAL == 0:
                val_ppl = math.exp(val_loss) if val_loss < 10 else float("inf")
                log_stat(f"Validation @ step {step}: loss={val_loss:.4f}, ppl={val_ppl:.2f}")
                upload_stats_to_hub(f"validation step {step}")
        else:
            val_loss = last_loss

        if step % LOG_CFG["checkpoint_interval"] == 0 and step != max_steps:
            save_checkpoint(
                model, optimizer, scheduler, step, samples_seen,
                val_loss if isinstance(val_loss, float) and val_loss else total_loss_window,
            )

        if step == max_steps:
            final_loss = total_loss_window / max(1, LOCAL_GRAD_ACCUM_STEPS)
            save_checkpoint(model, optimizer, scheduler, step, samples_seen, final_loss, final=True)
            if GLOBAL_ORDINAL == 0:
                log_stat(f"Training complete at step {step}")
                upload_stats_to_hub(f"training complete step {step}")

    xm.rendezvous("training_complete")

In [9]:
# 10. Launch training (single TPU)
train_worker(0)

RuntimeError: TPU initialization failed: Invalid --2a886c8_slice_builder_worker_addresses specified. Expected 4 worker addresses, got 1.

## Operational notes

1. Set the Colab runtime to **TPU** before importing `torch_xla`.
2. The two packed binary files are downloaded into `/content/project-intelligence-data` by default and memory-mapped during training. For very large files, keep them on Colab's local disk rather than a network-mounted filesystem when possible.
3. The default multi-core mode preserves the configured global effective batch by dividing accumulation across TPU replicas.
4. Set `MAX_STEPS_OVERRIDE = 1` or `2` for a compilation/checkpoint smoke test before committing to the full run.
5. Every checkpoint is saved locally and uploaded to `sovannpanhaseng/alpha-intel` under `checkpoints/` as `project-intelligence_step_<step>_<UTC timestamp>.pt` (final: `..._final_step_...`). On every start the notebook lists that folder, downloads the newest checkpoint (by timestamp in the name) and resumes from it. Set `KEEP_LAST_N_REMOTE_CHECKPOINTS` in cell 7 to prune old ones.
6. Authentication uses the Hugging Face notebook login flow; the account must have write access to the checkpoint repository.
7. The optimizer is the regular `torch.optim.AdamW`; no 8-bit, paged, CPU-offloaded, or custom AdamW implementation is used.
8. The notebook intentionally removes CUDA-only mechanisms such as TF32 flags, CUDA FlashAttention probing, CUDA memory queries, and CUDA UVM/8-bit optimizer paths; those are not the right execution primitives for TPU/XLA.
9. Training stats are pushed to `training_stats/training_stats_<session UTC timestamp>.txt` on the same repo after each validation, each checkpoint and at the end. Every line carries a UTC timestamp. Stats are never downloaded on restart; each restart begins a new stats file.

### References
- PyTorch/XLA TPU runtime and multi-device launch: https://docs.pytorch.org/xla/release/r2.8/learn/xla-overview.html
- PyTorch/XLA device loading and synchronized optimizer updates: https://docs.pytorch.org/xla/master/learn/pytorch-on-xla-devices.html
- PyTorch/XLA checkpoint saving: https://docs.pytorch.org/xla/master/learn/migration-to-xla-on-tpus.html
- Hugging Face Hub file download API: https://huggingface.co/docs/huggingface_hub/guides/download

In [ ]:
# 11. Notebook self-audit: naming, optimizer, and checkpoint/stats filename scheme.
assert RUN_NAME == "project-intelligence"
assert MODEL_NAME == "project-intelligence"
assert "project-intelligence" in str(DATA_ROOT)
assert "project-intelligence" in str(CHECKPOINT_ROOT)
assert HF_CHECKPOINT_REPO_ID == "sovannpanhaseng/alpha-intel"

_a, _b = parse_checkpoint_name(checkpoint_filename(500)), parse_checkpoint_name(checkpoint_filename(500, final=True))
assert _a and _a["step"] == 500 and not _a["final"], "periodic checkpoint name does not round-trip"
assert _b and _b["step"] == 500 and _b["final"], "final checkpoint name does not round-trip"
assert parse_checkpoint_name("latest_project-intelligence.pt") is None
assert LOG_FILE.name == f"training_stats_{SESSION_ID}.txt"
print("Naming, checkpoint-name, and stats-name audit: PASS")